In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             f1_score, precision_recall_curve, average_precision_score)

In [2]:
DATA_DIR = Path("../logistic_regression_modeling/final_data")

df = pd.read_csv(DATA_DIR / "cleaned_2019.csv")
df = df[[i for i in df.columns if 'original' not in i]]
df = df[[i for i in df.columns if 'qa' not in i]]
df = df[[i for i in df.columns if 'income' not in i]]
df = df.iloc[:, :-4]
df = df.drop(columns=['fd_label', 'label_status', 'label_reason'])

print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '..\\logistic_regression_modeling\\final_data\\cleaned_2019.csv'

In [ ]:
TARGET = "FD_2"

ID_COLS = ["tract_id", "State", "County", "meta_dataset_version",
           "meta_population_census_year", "meta_tract_vintage"]

LEAKAGE_COLS = ["clean__low_access_population_1_20",
                "clean__LATracts_half",
                "clean__LATracts1"]

# The 10 features in the final logistic regression model (model2 / full_model)
LR_FEATURES = ["clean__occupied_housing_units_total", "clean__NUMGQTRS",
               "clean__PovertyRate", "clean__MedianFamilyIncome", "clean__TractLOWI",
               "clean__TractKids", "clean__TractHispanic", "clean__TractHUNV",
               "clean__TractSNAP", "clean__LATractsVehicle_20"]

USE_LR_FEATURES = False

if USE_LR_FEATURES:
    FEATURES = LR_FEATURES
else:
    FEATURES = [c for c in df.columns if c not in ID_COLS + LEAKAGE_COLS + [TARGET]]

print(f"{len(FEATURES)} features:")
FEATURES

In [ ]:
df[TARGET].value_counts(normalize=True)

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

In [ ]:
baseline = clone(pipeline).fit(X_train, y_train)

train_ap = average_precision_score(y_train, baseline.predict_proba(X_train)[:, 1])
test_ap  = average_precision_score(y_test,  baseline.predict_proba(X_test)[:, 1])
print(f"Baseline train AP: {train_ap:.3f}")
print(f"Baseline test AP:  {test_ap:.3f}")

In [ ]:
param_dist = {
    "rf__n_estimators": [200, 300, 500],
    "rf__max_depth": [None, 10, 15, 20, 30],
    "rf__min_samples_leaf": [1, 2, 5, 10],
    "rf__max_features": ["sqrt", "log2", 0.5],
    "rf__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring="average_precision",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best settings:", search.best_params_)
print(f"Best cross-validated AP: {search.best_score_:.3f}")

best_model = search.best_estimator_

In [ ]:
cv_probs = cross_val_predict(best_model, X_train, y_train, cv=3,
                             method="predict_proba", n_jobs=-1)[:, 1]

thresholds = np.arange(0.05, 0.90, 0.05)
thresh_table = pd.DataFrame({
    "threshold": thresholds.round(2),
    "accuracy": [accuracy_score(y_train, (cv_probs >= t).astype(int)) for t in thresholds],
    "f1":       [f1_score(y_train, (cv_probs >= t).astype(int)) for t in thresholds],
})

best_threshold = float(thresh_table.loc[thresh_table["accuracy"].idxmax(), "threshold"])
print("Chosen threshold (max accuracy):", best_threshold)
thresh_table

In [ ]:
test_probs = best_model.predict_proba(X_test)[:, 1]

testData = df.loc[X_test.index].copy()
testData["pred_prob"] = test_probs
testData["pred_class"] = (test_probs >= best_threshold).astype(int)

print("Accuracy :", accuracy_score(y_test, testData["pred_class"]))
print(confusion_matrix(y_test, testData["pred_class"]))  # rows = actual, cols = predicted
print()
print(classification_report(y_test, testData["pred_class"], digits=3))

In [ ]:
testData["risk"] = pd.cut(
    testData["pred_prob"],
    bins=[0, 0.33, 0.67, 1],
    labels=["low-risk", "medium-risk", "high-risk"],
    include_lowest=True,
)
print(testData["risk"].value_counts())

In [ ]:
precision_test, recall_test, _ = precision_recall_curve(y_test, test_probs)
ap_test = average_precision_score(y_test, test_probs)

plt.plot(recall_test, precision_test)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Random Forest Precision-Recall Curve (AP = {ap_test:.3f})")
plt.savefig("rf_precision_recall_test.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sample = X_test.sample(n=min(5000, len(X_test)), random_state=42)

imp = permutation_importance(
    best_model, sample, y_test.loc[sample.index],
    scoring="average_precision", n_repeats=5, random_state=42, n_jobs=-1,
)

importance = (pd.Series(imp.importances_mean, index=FEATURES)
                .sort_values(ascending=True))

importance.plot.barh(figsize=(8, max(4, len(FEATURES) * 0.3)))
plt.xlabel("Drop in average precision when shuffled")
plt.title("Permutation Feature Importance")
plt.savefig("rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

importance.sort_values(ascending=False)

In [ ]:
final_model = clone(best_model).fit(X, y)

bundle = {
    "model": final_model,
    "features": FEATURES,
    "threshold": best_threshold,
    "best_params": search.best_params_,
}

with open("rf_model.pkl", "wb") as f:
    pickle.dump(bundle, f)

size_mb = Path("rf_model.pkl").stat().st_size / 1e6
print(f"Saved rf_model.pkl ({size_mb:.1f} MB)")